# LLM Model Testing with Ollama
- This notebook uses ollama models to test their setup in notebooks and ease of use for different models

## Ollama Setup
- Download and set up ollama
- Start the ollama server
- Pull a model
- Inference with the model

In [2]:
%%bash
nvidia-smi

Tue Nov  4 18:32:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   35C    P5              4W /  100W |     507MiB /   6141MiB |     44%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+------------------------------------------------------------------------------

In [3]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0% 11.3% 13.3% 14.7% 17.0% 18.5%9.2%      24.9% 25.8% 27.0% 30.8%32.9%5.4%% 57.4%%8% 65.4% 70.0% 84.8%2%4% 93.9% 97.4%.0%5%
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [5]:
import requests
import time
import subprocess

## Initialise Ollama Server

In [6]:
# Start Ollama server in background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait until server responds
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=1)
        if r.ok:
            print("[ollama server] Ollama server ready")
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)

[ollama server] Waiting for server... (1/30)
[ollama server] Ollama server ready


## Download Ollama models

In [ ]:
# List of models you want to download
models = [
    "qwen3:8b",
    "mistral:7b",
    "llama3.1:8b",
    "gemma3:12b"
]

for model in models:
    print(f"[model download] Downloading {model} ...")
    try:
        # Run the Ollama pull command
        result = subprocess.run(
            ["ollama", "pull", model],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True
        )
        print(result.stdout)
        print(f"[model download] Successfully downloaded {model}\n")
    except subprocess.CalledProcessError as e:
        print(f"[model download] Failed to download {model}")
        print(e.stderr)


Ollama server ready ✅
⬇️ Downloading qwen3:8b ...

✅ Successfully downloaded qwen3:8b

⬇️ Downloading mistral:7b ...

✅ Successfully downloaded mistral:7b

⬇️ Downloading llama3.1:8b ...

✅ Successfully downloaded llama3.1:8b

⬇️ Downloading gemma3:12b ...

✅ Successfully downloaded gemma3:12b



In [ ]:
# Test URL
url = "http://localhost:11434/api/tags"

print("🔍 Checking if Ollama server is running...")

for i in range(30):  # wait up to ~30 seconds
    try:
        r = requests.get(url, timeout=2)
        if r.ok:
            print("[ollama server] Ollama server is running!")
            print("Available models:")
            k = r.json()["models"]
            for a, m in enumerate(k):
                print(a, m["name"])
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)
else:
    print("[ollama server] Ollama server not responding.")


🔍 Checking if Ollama server is running...
[ollama server] Ollama server is running!
Available models:
0 gemma3:12b
1 llama3.1:8b
2 mistral:7b
3 qwen3:8b


## Test With Model

In [22]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='gemma3:12b', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

Okay, let's break down why the sky is blue! It's a classic question with a surprisingly fascinating answer rooted in physics. Here's the explanation, broken down into understandable parts:

**1. Sunlight is a Mixture of Colors**

*   Sunlight doesn't just look white to us. It's actually composed of *all* the colors of the rainbow! You can see this when sunlight passes through a prism (or sometimes when water droplets in the air create a rainbow). These colors have different wavelengths.
    *   **Wavelengths:**  Think of wavelengths like the distance between the peaks of a wave.  Blue and violet light have shorter wavelengths, while red and orange light have longer wavelengths.

**2.  Entering the Atmosphere:  A Collision Course**

*   When sunlight enters the Earth's atmosphere, it bumps into tiny air molecules (mostly nitrogen and oxygen). This interaction is crucial.

**3.  Rayleigh Scattering - The Key Process**

*   **What it is:** This is the key phenomenon at play. Rayleigh scat